# 06.2 - Loss Functions

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

A loss function measures the gap between a model's prediction and the true target, converting 'how wrong am I?' into a single scalar that gradient descent can minimize.

## 2. Why Does This Matter?

Choosing the wrong loss means optimizing for the wrong thing — MSE on classification or cross-entropy on regression produces nonsense. The loss is the contract between your model and your objective.

## 3. Prerequisites

- Unit 06.1 (perceptron, activations), basic probability

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement MSE, MAE, Huber, BCE, and categorical cross-entropy
- Choose the right loss for a task and explain why
- Debug NaN losses and mismatched loss/metric behavior

## 5. Mental Model

A loss function is a scoring rubric: the lower the score, the better. Different tasks need different rubrics — 'how far off?' (regression) vs 'how confident in the wrong answer?' (classification).

- MSE: `L = (1/n) Σ(ŷ - y)²`
- BCE: `L = -[y·log(ŷ) + (1-y)·log(1-ŷ)]`
- CE: `L = -Σ yᵢ·log(ŷᵢ)`


## 6. Backend


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
print("Backend ready:", matplotlib.get_backend())


Backend ready: Agg


## 7. Implement the Core Losses from Scratch


In [2]:
def mse(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

def mae(y_pred, y_true):
    return np.mean(np.abs(y_pred - y_true))

def bce(y_pred, y_true, eps=1e-8):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def cross_entropy(y_pred, y_true, eps=1e-8):
    y_pred = np.clip(y_pred, eps, 1.0)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

# Regression demo
y_reg = np.array([1.0, 2.5, 3.0, 4.2])
p_reg = np.array([1.1, 2.0, 3.3, 3.9])
print(f"MSE: {mse(p_reg, y_reg):.4f}")
print(f"MAE: {mae(p_reg, y_reg):.4f}")

# Binary classification demo
y_bin = np.array([0, 1, 1, 0])
p_bin = np.array([0.1, 0.9, 0.8, 0.2])
print(f"BCE: {bce(p_bin, y_bin):.4f}")

# Multi-class demo (one-hot)
y_multi = np.array([[1,0,0],[0,1,0],[0,0,1]])
p_multi = np.array([[0.7,0.2,0.1],[0.1,0.8,0.1],[0.2,0.1,0.7]])
print(f"CE:  {cross_entropy(p_multi, y_multi):.4f}")


MSE: 0.1100
MAE: 0.3000
BCE: 0.1643
CE:  0.3122


## 8. Why MSE Fails for Binary Classification

Near confident correct predictions, the gradient of MSE shrinks toward 0, killing learning. Cross-entropy keeps a strong gradient signal for wrong confident predictions.


In [3]:
# Compare gradients for a single sample, true label = 1
p = np.linspace(0.01, 0.99, 100)
y = 1.0
# d(BCE)/dp = -(y/p - (1-y)/(1-p))
grad_bce = -(y / p - (1 - y) / (1 - p))
# d(MSE)/dp = 2*(p - y)
grad_mse = 2 * (p - y)

plt.figure(figsize=(8, 4))
plt.plot(p, grad_bce, label="BCE gradient", linewidth=2)
plt.plot(p, grad_mse, label="MSE gradient", linewidth=2)
plt.axhline(0, color='gray', lw=0.5)
plt.xlabel("prediction p (true label 1)")
plt.ylabel("gradient magnitude")
plt.title("Gradient strength: BCE vs MSE near confident wrong predictions")
plt.legend()
plt.savefig('_tmp_loss_grad.png', dpi=80)
plt.close()
print("At p=0.1 (confidently wrong): BCE grad =", round(grad_bce[9], 3), " MSE grad =", round(grad_mse[9], 3))
print("MSE gradient collapses for confident wrong predictions — this is why MSE is bad for classification.")


At p=0.1 (confidently wrong): BCE grad = -10.092  MSE grad = -1.802
MSE gradient collapses for confident wrong predictions — this is why MSE is bad for classification.


## 9. Huber Loss vs MSE on Outliers

Huber combines the robustness of MAE with the smooth gradients of MSE near zero.


In [4]:
def huber(y_pred, y_true, delta=1.0):
    diff = np.abs(y_pred - y_true)
    quad = 0.5 * diff**2
    lin = delta * (diff - 0.5 * delta)
    return np.mean(np.where(diff <= delta, quad, lin))

np.random.seed(1)
n = 200
x = np.random.uniform(-3, 3, n)
y = 0.5 * x + np.random.normal(0, 0.3, n)
# Inject outliers
y[:10] += np.random.normal(0, 20, 10)

print(f"MSE with outliers:  {mse(y, 0.5*x):.3f}")
print(f"Huber with outliers: {huber(0.5*x, y):.3f}")
print("\nHuber is far less sensitive to the outliers (robust regression).")


MSE with outliers:  9.948
Huber with outliers: 0.630

Huber is far less sensitive to the outliers (robust regression).


## 10. Numerical Stability: log of zero

Without epsilon, `log(0)` → -inf or NaN. Clipping predictions guarantees finite loss.


In [5]:
p_bad = np.array([0.0, 1.0])  # exactly 0 and 1
y_bad = np.array([1.0, 1.0])
try:
    raw = np.mean(- (y_bad * np.log(p_bad)))
    print("raw (no guard):", raw)
except Exception as e:
    print("raw (no guard) error:", e)

p_safe = np.clip(p_bad, 1e-8, 1 - 1e-8)
print("clipped via BCE helper:", bce(p_bad, y_bad))
print("\nClipping predictions before log prevents NaN/inf.")


raw (no guard): inf
clipped via BCE helper: 9.210340376976184

Clipping predictions before log prevents NaN/inf.


C:\Users\PC\AppData\Local\Temp\ipykernel_10728\2108003218.py:4: RuntimeWarning: divide by zero encountered in log
  raw = np.mean(- (y_bad * np.log(p_bad)))


## 11. Decision Guidance

| Task | Recommend | Why | Avoid |
|---|---|---|---|
| Binary classification | BCE | Probabilistic, differentiable | MSE (vanishing gradients) |
| Multi-class | CE + softmax | Proper scoring rule | MSE on one-hot |
| Regression | MSE | Smooth, penalizes large errors | MAE when outliers dominate |
| Regression w/ outliers | Huber | Combines MSE + MAE | MSE on extreme outliers |
| Imbalanced | Weighted CE | Downweights majority | Unweighted CE |

## 12. Common Mistakes

- Using MSE for classification.
- Forgetting numerical stability (log of zero = NaN).
- Softmax + CE without understanding they are linked.
- Not normalizing by batch size (LR becomes batch-size sensitive).

## 13. Debugging / Troubleshooting

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Loss is NaN | log of 0 / very small pred | Print predictions | Add epsilon, clip |
| Loss stops decreasing | Saturated softmax | Inspect pred distribution | Label smoothing |
| Loss drops, accuracy doesn't | Mismatched loss/metric | Plot both | Verify loss matches task |
| Loss varies wildly | Small batch / high LR | Monitor variance | Bigger batch / lower LR |

## 14. When NOT to Use

- BCE for multi-class; use categorical cross-entropy.
- MSE where outliers should not dominate; use Huber.

## 15. Challenge

Add class weights to cross-entropy and show that a minority class is weighted up.


In [6]:
# Challenge: class-weighted cross-entropy
def weighted_ce(y_pred, y_true, class_weights, eps=1e-8):
    y_pred = np.clip(y_pred, eps, 1.0)
    per_sample = -np.sum(y_true * np.log(y_pred), axis=1)
    w = np.sum(y_true * np.array(class_weights).reshape(1, -1), axis=1)
    return np.mean(w * per_sample)

p = np.array([[0.9, 0.1], [0.3, 0.7], [0.8, 0.2]])
y = np.array([[1,0],[0,1],[0,1]])
print("Unweighted CE:", round(cross_entropy(p, y), 4))
print("Weighted CE (class1 x5):", round(weighted_ce(p, y, [1.0, 5.0]), 4))
print("\nWeighting errors on the minority class up improves learning on it.")


Unweighted CE: 0.6905
Weighted CE (class1 x5): 3.312

Weighting errors on the minority class up improves learning on it.


## 16. Closed-Book Recall

Without looking back:

1. Why does cross-entropy pair naturally with softmax?
2. What problem does MSE have for binary classification?
3. How does adding epsilon to log prevent NaN?
4. Why does weighted cross-entropy help imbalanced classes?

## 17. Teach-Back Questions

Explain to another person:

- The difference between MSE and cross-entropy.
- When you would choose Huber over MSE.

## 18. Summary

You implemented MSE, MAE, Huber, BCE, and cross-entropy from scratch, compared gradients, handled numeric stability, and explored class weighting.

## 19. Further Experiment

- Try label smoothing on cross-entropy.
- Verify CE gradient analytically vs numerically.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
